# Soft voting and out-of-fold stacking

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load cohort

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Generate out-of-fold predictions for training a logistic meta-learner

In [ ]:
from src.advanced_experiment import pipeline
from src.models.boosting import models
from src.models.ensembles import PrefitEnsemble
from sklearn.linear_model import LogisticRegression
members=[]; oof=[]
for name,est in models().items():
    if QUICK: est.set_params(n_estimators=20)
    m=pipeline(X,QUICK).set_params(model=est); p,r=group_cv(m,X,Y,G); oof.append(p); members.append(m.fit(X,Y)); print(name,r.mean(axis=0))
meta=LogisticRegression().fit(np.column_stack(oof),Y)

Compare ensemble probabilities on validation

In [ ]:
models_={'SoftVoting':PrefitEnsemble(members),'Stacking':PrefitEnsemble(members,meta)}
preds={k:m.predict_proba(x.iloc[tune])[:,1] for k,m in models_.items()}; r=pd.DataFrame([{'model':k,**scores(y.iloc[tune],p)} for k,p in preds.items()]); display(r); plot_curves(y.iloc[tune],preds); r.to_csv(OUT/'ensembles.csv',index=False)